# 09. Registro de afirmaciones (claims registry)

Este cuaderno sella el **registro auditable de afirmaciones** del
proyecto: cada cifra que la tesis puede citar, con la ruta JSON
exacta de la publicación de donde sale. No entrena ni recalcula
modelos ni portafolios: lee los punteros verificados byte a byte de
los cuadernos 02–08 bajo la misma maquinaria de publicación que el
08, y aborta ante cualquier discrepancia.

La única computación propia es una **verificación independiente**:
el conteo de pares colindantes prueba–entrenamiento tras el buffer
del arnés espacial del 04 se reconstruye desde los artefactos
publicados (asignaciones de fold, exclusiones de buffer y geometría
del universo) y debe coincidir con el reporte —si no, el build
aborta—. El documento ejecutivo `TESIS_DATA_POINTS.md` se genera
mecánicamente desde el registro: ningún número está escrito a mano.

In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, JSON

BASE = Path.cwd().resolve()
PROJECT = next(p / "4_proyecto" for p in [BASE, *BASE.parents] if (p / "4_proyecto/V3/data").is_dir())
V3_DIR = PROJECT / "V3"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from V3.src.c9_analysis import CONFIG, load_inputs, analyze
from V3.src.c9_publication import stage_results

data = load_inputs(V3_DIR)
result = analyze(data, verbose=True)
report = result["report"]
display(pd.DataFrame([{
    "contract_id": report["contract_id"],
    "analysis_version": report["schema_version"],
    "shared_validation_analysis_id":
        report["upstream"]["shared_validation_analysis_id"],
    "reads_verified_only":
        report["scope"]["reads_verified_publications_only"],
    "trains_no_models": report["scope"]["trains_no_models"],
}]))

[c9] certificacion: 0.1s


[c9] claims: 0.1s


,contract_id,analysis_version,shared_validation_analysis_id,reads_verified_only,trains_no_models
0,6e586b3d9a0bd8b844d9cf5bc714f0e583fc2a6936f8cd...,v3-claims-1,ed7bf47bafab10ca2b3144e9d38f1d06e5024ee9ba6af3...,True,True


## 1. Linaje de publicaciones

Los cuadernos 01–08 que alimentan el registro, con su esquema,
`analysis_id` y el sha256 del puntero leído. La auditoría ETL (01)
no es un reporte de motor: no lleva `analysis_id` y se registra por
su hash de archivo.

In [2]:
display(result["lineage_df"])

,cuaderno,notebook,puntero,puntero_sha256,schema_version,analysis_id,contract_id,nota
0,etl,01_ETL_Reproducible.ipynb,audits/01_etl.json,198a41c57443e8fcabcf9233f592f6cdd2f5b8a4e7e0b9...,None,None,6e586b3d9a0bd8b844d9cf5bc714f0e583fc2a6936f8cd...,auditoría ETL sin analysis_id: es el archivo d...
1,eda,02_EDA_y_Equidad.ipynb,audits/02_eda.json,a17c6003a544b47571a4ca62d7b8630e973a49fc5d5c5c...,v3-eda-2,d77ab12f05e9025f438921b34e8688e1a7562826296427...,6e586b3d9a0bd8b844d9cf5bc714f0e583fc2a6936f8cd...,None
2,features,03_Feature_Engineering.ipynb,audits/03_feature_engineering.json,9d3149fe44145e2f5484deb03dc3ac74cfd2910a0c7ca9...,v3-features-2,7565adda3bc4eb91514247d2a1213f0b1c934821e762ac...,6e586b3d9a0bd8b844d9cf5bc714f0e583fc2a6936f8cd...,None
3,validation,04_Diseno_Validacion_Backtest.ipynb,audits/04_backtest_design.json,61beeaa81a0a013fea1d2f273d14217c8dbe44fc1a8217...,v3-backtest-4,ed7bf47bafab10ca2b3144e9d38f1d06e5024ee9ba6af3...,6e586b3d9a0bd8b844d9cf5bc714f0e583fc2a6936f8cd...,None
4,motor1,05_Motor1_Baseline_Estadistico.ipynb,audits/05_motor1_baseline.json,bfc72c5ebb4f824a649cda661d9d9a95654f1d1f8c6a1d...,v3-motor1-3,879ae45d46b56fc5504c9817be154eee0e78ab0e13ddfd...,6e586b3d9a0bd8b844d9cf5bc714f0e583fc2a6936f8cd...,None
5,motor2,06_Motor2_Transformer_Honesto.ipynb,audits/06_motor2_transformer.json,50212264e2f0fa885a9c21ca2e3a55d3c59ea70323d590...,v3-motor2-3,4027385dff30ab79689dfdeb3c468260176080e83b1abd...,6e586b3d9a0bd8b844d9cf5bc714f0e583fc2a6936f8cd...,None
6,motor3,07_Motor3_Grafo_y_Genetico.ipynb,audits/07_motor3_prescriptivo.json,1dc9bb280efb746fc2711f211f24f722352111d2513319...,v3-motor3-4,3aa53121add31f2ea2381259d0b44fa7f603fa758b4a73...,6e586b3d9a0bd8b844d9cf5bc714f0e583fc2a6936f8cd...,None
7,comparacion,08_Evaluacion_Comparativa.ipynb,audits/08_comparacion_portafolios.json,adcb5ff88a0760f15d3679521ffa3b04c05c6a8ca17aa7...,v3-comparacion-2,fdba0469ee4d2a42ce8bc5f75d53e95930207bfe3152fa...,6e586b3d9a0bd8b844d9cf5bc714f0e583fc2a6936f8cd...,None


## 2. Afirmaciones registradas

Cada afirmación declara su estatus —`hecho_verificado`,
`comparacion_descriptiva`, `resultado_negativo` o
`advertencia_de_alcance`—, sus fuentes (reporte, `analysis_id` y
ruta JSON) y sus salvedades, que nunca están vacías.

In [3]:
display(result["claims_df"][["id", "tema", "estatus", "enunciado"]])
for _, row in result["claims_df"].iterrows():
    display(Markdown(f"**{row['id']} · {row['tema']}** — "
                     f"salvedades:"))
    for s in json.loads(row["salvedades_json"]):
        display(Markdown(f"- {s}"))

,id,tema,estatus,enunciado
0,C01,cobertura_poblacional,comparacion_descriptiva,"A B=30, la población nueva en áreas de captura..."
1,C02,cobertura_poblacional,comparacion_descriptiva,"A B=50, la población nueva en áreas de captura..."
2,C03,divergencia_territorial,comparacion_descriptiva,Los motores seleccionan portafolios territoria...
3,C04,tension_similitud_equidad,comparacion_descriptiva,"A B=30, la escolaridad media de las AGEBs sele..."
4,C05,tiempos_greedy_vs_ga,hecho_verificado,Con la misma función objetivo y reglas de fact...
5,C06,objetivo_greedy_vs_ga,hecho_verificado,En las 7 configuraciones evaluadas el greedy a...
6,C07,dictamen_motor2,hecho_verificado,El FT-Transformer no justifica su complejidad:...
7,C08,arnes_espacial,hecho_verificado,El esquema primario deja 0 pares colindantes p...
8,C09,compuerta_ledoit_wolf,hecho_verificado,"Con n/p entre 2.00 y 2.56, el encogimiento Led..."
9,C10,remocion_techo_analogos,resultado_negativo,El techo por análogos —densidad histórica de p...


**C01 · cobertura_poblacional** — salvedades:

- Población dentro de áreas de captura de 500 m con interpolación areal; no es demanda observada

- Los motores 1 y 2 ya no usan techos por AGEB

- El Motor 3 optimiza afinidad por saltos a Golden; sus métricas de cercanía están por construcción

...

- Los números citados son de la publicación histórica v3-motor3-3 (analysis_id 69a83e61…); no se recalculan

- La nueva afinidad 1/(1+saltos) es determinista pero hace que la cercanía a Golden esté optimizada por construcción

- La afinidad estructural no es demanda ni evidencia causal

**C12 · consistencia_publicaciones** — salvedades:

- La consistencia verifica reproducibilidad de métricas publicadas; no valida los supuestos del contrato

## 3. Certificación del arnés espacial

El esquema primario `fold_spatial_block` elimina la colindancia
prueba–entrenamiento con un buffer de 500 m sobre la distancia
poligonal. La columna `recomputado_after` viene de la verificación
independiente de este cuaderno (adyacencia poligonal ≤ 1 m
recalculada desde los artefactos publicados): debe ser 0 en todos
los folds e igual a lo publicado. La garantía cubre colindancia
poligonal y buffer de 500 m; no descarta autocorrelación espacial a
mayores distancias.

In [4]:
cert = report["certificacion_arnes_espacial"]
display(pd.Series({
    "certificado": cert["certificado"],
    "suma_pares_after_buffer": cert["suma_pares_after_buffer"],
    "min_train_test_distance_m": cert["min_train_test_distance_m"],
    "buffer_m": cert["buffer_m"],
    "n_blocks": cert["n_blocks"],
    "blocks_contiguous_share": cert["blocks_contiguous_share"],
    "n_adjacency_components": cert["n_adjacency_components"],
    "recomputo_coincide": cert["recomputo"]["coincide_con_publicado"],
}, name="valor").to_frame())
display(pd.DataFrame(cert["folds"]))
display(Markdown("**Referencia optimista** "
                 "(`fold_groupkfold_ageb`, sin buffer): pares "
                 f"colindantes {cert['referencia_optimista_groupkfold']['pares_por_fold']}, "
                 f"suma {cert['referencia_optimista_groupkfold']['suma_pares']}."))

,valor
certificado,True
suma_pares_after_buffer,0
min_train_test_distance_m,500.583149
buffer_m,500.0
n_blocks,15
blocks_contiguous_share,1.0
n_adjacency_components,1
recomputo_coincide,True


,fold,touching_train_test_pairs_before_buffer,touching_train_test_pairs_after_buffer,min_train_test_distance_m,recomputado_after,recomputado_before
0,0,95,0,500.583149,0,95
1,1,57,0,508.322951,0,57
2,2,59,0,502.632218,0,59
3,3,50,0,505.049535,0,50
4,4,59,0,506.370875,0,59


**Referencia optimista** (`fold_groupkfold_ageb`, sin buffer): pares colindantes {'0': 220, '1': 232, '2': 206, '3': 214, '4': 216}, suma 1088.

## 4. Encuadres corregidos

Dos encuadres frecuentes que el registro corrige con la evidencia
publicada: «Ridge domina numéricamente al FT-Transformer» y el
cociente de tiempos totales no emparejados del GA frente al greedy.

In [5]:
for e in report["encuadres_corregidos"]:
    display(Markdown(
        f"**{e['id']}** — afirmación original: *{e['afirmacion_original']}* "
        f"({e['estatus_afirmacion_original']}).<br/>"
        f"Encuadre correcto: {e['encuadre_correcto']}"))
    display(pd.DataFrame(
        pd.Series(e["evidencia"], dtype=object), columns=["valor"]))

**ridge_vs_ft_transformer** — afirmación original: *Ridge domina numéricamente al FT-Transformer* (no_sustentada).<br/>Encuadre correcto: Ridge tiene el RMSE medio más bajo por esa diferencia mínima, pero el FT-Transformer muestra R² (0.215 vs 0.194) y Spearman (0.585 vs 0.531) ligeramente superiores: ninguno de los dos domina; el resultado es un empate práctico y la complejidad del Transformer no se justifica.

,valor
rmse_ridge,0.41952
rmse_ft_transformer,0.420556
delta_rmse,0.001036
condition_c_seed_std_mean,0.031288
r2_ridge,0.193849
r2_ft_transformer,0.214797
spearman_ridge,0.531347
spearman_ft_transformer,0.585396


**tiempos_totales_vs_emparejados** — afirmación original: *El greedy es ~10^4 veces más rápido que el GA (0.12 s vs 1,303 s)* (no_sustentada).<br/>Encuadre correcto: El encuadre defendible es el speedup emparejado por configuración (misma función objetivo y factibilidad): GA medio por semilla / greedy por corrida; los totales no emparejados no definen un cociente.

,valor
greedy_total_wall_s,0.123414
ga_total_wall_s,1302.635813
porque_no_es_emparejado,el greedy corre una vez por configuración; el ...
speedup_emparejado_primario,3193.255962


## 5. Ledger de límites

Unión de las listas `pending_validation` de los reportes 04–08 con
su origen: toda cifra del registro hereda estas limitaciones.

In [6]:
display(pd.DataFrame(report["limitations_ledger"]))

,origen,nota
0,validation,Engine rankings themselves (cuadernos 05/06/07)
1,validation,Sensitivity of in_domain to the Mahalanobis qu...
2,validation,Sensitivity of metrics to the label quantile
3,validation,Alternative NDCG gains (binary vs linear vs log)
4,validation,Buffer distance sensitivity (500 m declared)
5,motor1,Sensitivity of deployment ranking to domain/re...
6,motor2,"Analog-ceiling backtest is removal evidence, n..."
7,motor2,Capacity study says nothing about latent demand
8,motor2,"RF is the deployment model by parsimony, not b..."
9,motor2,Deployment predictions are extrapolation outsi...


## 6. Documento ejecutivo y cierre

`TESIS_DATA_POINTS.md` se genera desde el registro con un léxico
controlado (sin «domina», «ganador», «demuestra», «óptimo» sin
negación, ni similitud/cobertura presentadas como demanda) y se
publica como output sellado más una copia atómica en
`docs/TESIS_DATA_POINTS.md` tras la activación del puntero.

In [7]:
doc = report["document"]
display(pd.Series({
    "documento": doc["output"],
    "sha256": doc["sha256"],
    "copia_docs": doc["docs_copy_path"],
    "lexico_verificado": doc["forbidden_lexicon_checked"],
    "n_claims": len(report["claims"]),
}, name="valor").to_frame())
display(Markdown(result["documento"]))
report_path = stage_results(V3_DIR, data, result)
display(Markdown("**Cierre.** Registro de afirmaciones sellado: "
                 "toda cifra citable queda ligada a su publicación "
                 "de origen."))
display(JSON({"c9_report_path": report_path.relative_to(V3_DIR).as_posix()}))

,valor
documento,TESIS_DATA_POINTS.md
sha256,2889271334af26e58a5447009e7860e2589f4ea472bfa8...
copia_docs,docs/TESIS_DATA_POINTS.md
lexico_verificado,True
n_claims,12


# TESIS_DATA_POINTS — registro de afirmaciones con fuente

Documento generado mecánicamente desde el registro sellado `v3-claims-1` (analysis_id `acd556d569dfef3511f037479780acc3b54ed5f850aea2670b279bbf32b96422`). Cada cifra lleva en el registro la ruta JSON de su fuente; este texto solo las formatea. Nada aquí es demanda observada: las cifras de cobertura son población dentro de áreas de captura de 500 m con interpolación areal.

## 1. Cobertura poblacional por motor y presupuesto

A B=30: el Motor 1 cubre 202,337 habitantes nuevos en 21 AGEBs, el Motor 2 282,102 en 25 y el Motor 3 310,288 en 28, siempre bajo el mismo contrato de despliegue y sin techos por AGEB. Respecto a la publicación previa con techos, los portafolios B=30 de los motores 1 y 2 crecieron +48,528 y +6,159 habitantes: el aumento proviene de retirar un tope que el backtest no validó, no de un método mejor. Es población en captura, no demanda observada, y las diferencias son entre portafolios heurísticos sin prueba estadística.

| Motor | B | Población nueva | % pobl. | AGEBs | 1/2/3+ |
|---|---|---|---|---|---|
| Motor 1 | 15 | 113,528 | 1.42% | 10 | 8/1/1 |
| Motor 2 | 15 | 136,490 | 1.71% | 12 | 10/1/1 |
| Motor 3 | 15 | 166,731 | 2.09% | 14 | 13/1/0 |
| Motor 1 | 30 | 202,337 | 2.54% | 21 | 15/5/1 |
| Motor 2 | 30 | 282,102 | 3.54% | 25 | 21/3/1 |
| Motor 3 | 30 | 310,288 | 3.89% | 28 | 26/2/0 |
| Motor 1 | 50 | 300,803 | 3.78% | 33 | 20/11/2 |
| Motor 2 | 50 | 428,401 | 5.38% | 41 | 33/7/1 |
| Motor 3 | 50 | 481,554 | 6.04% | 46 | 42/4/0 |

Los portafolios divergen territorialmente: a B=30 el Jaccard de sitios es 0.176 (M1–M2), 0.091 (M1–M3) y 0.250 (M2–M3), y de AGEBs 0.150, 0.089 y 0.262; 18 sitios aparecen en al menos 2 motores y 4 en los 3. La divergencia es descriptiva: los objetivos difieren por diseño.

## 2. Navaja de Ockham

**Motor 3 — greedy frente a GA.** Con la misma función objetivo y las mismas reglas de factibilidad, en la configuración primaria (w=0.5, B=30) el greedy toma 0.016 s por corrida frente a 51.4 s por semilla del GA (3,193x; rango emparejado 2,786x–4,422x entre configuraciones). Los totales no emparejados —greedy 0.123 s en una pasada por configuración, GA 1,302.6 s en 3–5 semillas por configuración— no definen un cociente. En el objetivo, el greedy iguala o supera a la mejor corrida del GA en 7 de 7 configuraciones: en 5 configuraciones la ventaja del greedy excede la variación entre semillas del GA; en 2 la diferencia es menor que esa variación (empate práctico); el GA nunca supera al greedy; la mejor corrida del GA queda entre 0.0009 y 0.0158 por debajo del greedy en la función objetivo. Es un hallazgo empírico: la garantía (1−1/e) de cobertura submodular no aplica formalmente bajo contigüidad y espaciamiento, y el GA es una heurística sin garantía de optimalidad.

**Motor 2 — modelos tabulares.** En el esquema espacial primario el RMSE medio es 0.5444 (constante), 0.4195 (ridge), 0.4288 (bosque aleatorio) y 0.4206 (FT-Transformer). Frente al mejor control (Ridge) el Transformer no mejora: su diferencia de RMSE es de +0.0010 a favor de Ridge, que equivale a 0.033 veces la variación entre semillas (0.0313). Ridge tiene el RMSE medio más bajo por esa diferencia mínima, pero el FT-Transformer muestra R² (0.215 vs 0.194) y Spearman (0.585 vs 0.531) ligeramente superiores: ninguno de los dos domina; el resultado es un empate práctico y la complejidad del Transformer no se justifica. El dictamen mecánico lo retira del despliegue y el bosque aleatorio se despliega por parsimonia y estabilidad, no por superioridad acreditada: RF vs Ridge difiere en +0.0092 de RMSE medio con R² espacial ≤ ~0.22 en todos los modelos.

## 3. Certificación del arnés espacial

El esquema primario `fold_spatial_block` deja 0 pares colindantes prueba–entrenamiento tras el buffer de 500 m en los 5 folds (antes del buffer: [95, 57, 59, 50, 59]); la distancia mínima retenida es 500.6 m y los 15 bloques forman un solo componente de adyacencia (1) con contigüidad 1.00. El conteo fue recomputado de forma independiente desde los artefactos publicados y coincide con el reporte. La garantía es sobre colindancia poligonal (≤1 m) y buffer de 500 m; no descarta autocorrelación espacial a mayor distancia.

## 4. Otros hallazgos de método

- **Ledoit-Wolf (Motor 1):** con n/p de 2.00–2.56 el encogimiento (0.38–0.70) lleva el número de condición de 33–55 a 2.26–6.44 y la cobertura de decisión de 0.637 a 0.901; el NDCG@10 (0.673) no cambia respecto al coseno sin compuerta. El umbral chi-cuadrado es nominal bajo encogimiento.
- **Remoción del techo por análogos:** en 05 el análogo pierde en MAE (1.678 vs 1.256) y en Spearman (0.274 vs 0.307); en 06 pierde en MAE (1.438 vs 1.256) aunque su Spearman queda marginalmente por encima (0.312 vs 0.307). Reproduce densidad histórica de planeación, no demanda observada.
- **Retiro de Node2Vec:** la afinidad por embeddings no ordenaba por cercanía al núcleo (histórico v3-motor3-3, no recalculado: Spearman con −saltos -0.095 / -0.095 con q=0.5; razón de norma del centroide 0.735). La afinidad vigente es determinista, 1/(1+saltos a Golden), y por ello la cercanía a Golden del portafolio está optimizada por construcción —no es evidencia independiente—.
- **Tensión similitud–equidad:** a B=30 la escolaridad media de las AGEBs seleccionadas es 13.02 (M1), 11.83 (M2) y 11.64 (M3) frente a 12.23 en todos los sitios; la fracción fuera del dominio del 04 es 0.30 / 0.23 / 0.57. Son covariables ecológicas de AGEB, sin lectura causal.

## 5. Salvedades que deben acompañar cada cifra

- (C01, C02) Población dentro de áreas de captura de 500 m con interpolación areal; no es demanda observada
- (C01, C02) Los motores 1 y 2 ya no usan techos por AGEB
- (C01, C02) El Motor 3 optimiza afinidad por saltos a Golden; sus métricas de cercanía están por construcción
- (C01, C02) Diferencias entre portafolios heurísticos a presupuesto fijo, sin prueba estadística
- (C03) Jaccard descriptivo sobre conjuntos de sitios y de AGEBs; sin umbral de 'alto/bajo' declarado
- (C03) Los objetivos de los motores difieren por diseño
- (C04) Covariables a nivel AGEB (lectura ecológica), sin causalidad
- (C04) Fuera de dominio no implica mal sitio: el dominio del 04 es el soporte de AGEBs con servicio
- (C04) La semejanza a entornos afluentes no es demanda
- (C05) Tiempos de pared de una sola máquina y una sola medición por configuración
- (C05) Speedup emparejado por configuración (misma f y mismas reglas); los totales no emparejados se reportan aparte y no definen un cociente
- (C05) El GA corre con los parámetros publicados del 07 (población 80, 120 generaciones)
- (C06) La garantía (1-1/e) de cobertura submodular no aplica formalmente bajo contigüidad y espaciamiento; es un hallazgo empírico, no un teorema
- (C06) El GA es una heurística sin garantía de optimalidad (3-5 semillas por configuración)
- (C06) El resultado depende de los parámetros GA declarados en el 07
- (C07) R^2 espacial <= ~0.22 en todos los modelos: ninguno es predictivamente fuerte
- (C07) RF vs Ridge es una diferencia pequeña (RF gana en 3 de 5 folds)
- (C07) 5 folds espaciales; variación entre semillas del Transformer documentada como caso de estudio
- (C07) La importancia predictiva no es causalidad
- (C08) La garantía es sobre colindancia poligonal (<=1 m) y buffer de 500 m en el universo de AGEBs evaluadas; no implica ausencia de autocorrelación espacial a mayores distancias.
- (C09) El umbral chi-cuadrado es nominal bajo encogimiento: las distancias de entrenamiento no son exactamente chi
- (C09) ndcg_cond > ndcg indica abstención en casos difíciles y se lee junto con decision_coverage
- (C09) No hay mejora de NDCG@10 respecto al coseno sin compuerta
- (C10) El backtest del techo es evidencia de la remoción, no una regla de asignación activa
- (C10) En 06 el Spearman del análogo queda marginalmente por encima del baseline aunque el MAE es peor; el resultado no se simplifica a 'perdió en todo'
- (C11) Los números citados son de la publicación histórica v3-motor3-3 (analysis_id 69a83e61…); no se recalculan
- (C11) La nueva afinidad 1/(1+saltos) es determinista pero hace que la cercanía a Golden esté optimizada por construcción
- (C11) La afinidad estructural no es demanda ni evidencia causal
- (C12) La consistencia verifica reproducibilidad de métricas publicadas; no valida los supuestos del contrato

### Límites registrados por los cuadernos 04–08

Citados textualmente de los reportes publicados (en inglés).

- [validation] Engine rankings themselves (cuadernos 05/06/07)
- [validation] Sensitivity of in_domain to the Mahalanobis quantile
- [validation] Sensitivity of metrics to the label quantile
- [validation] Alternative NDCG gains (binary vs linear vs log)
- [validation] Buffer distance sensitivity (500 m declared)
- [motor1] Sensitivity of deployment ranking to domain/reference quantiles beyond one-at-a-time
- [motor2] Analog-ceiling backtest is removal evidence, not an active allocation rule
- [motor2] Capacity study says nothing about latent demand
- [motor2] RF is the deployment model by parsimony, not by proven dominance; no model is predictively strong
- [motor2] Deployment predictions are extrapolation outside the observed activity scale
- [motor3] Affinity is a deterministic function of hops to the golden core: golden-proximity portfolio metrics are optimized by construction and are not independent evidence
- [motor3] Structural affinity is proximity to the golden core, not observed demand
- [motor3, comparacion] Areal interpolation and 500 m catchments are assumptions, not observed demand
- [motor3, comparacion] No street network, cost, or field validation
- [motor3] The GA is a heuristic without optimality guarantee; its best run is published only as a sensitivity portfolio
- [comparacion] Comparison is descriptive: no aggregate winner is declared
- [comparacion] Radar area is not a score; axes encode different objectives
- [comparacion] Motor 3 optimizes golden-hop affinity directly: its proximity-to-golden metrics are by construction, not independent evidence
- [comparacion] Motors 1 and 2 dropped analog ceilings; their portfolios changed relative to the previous publications
- [comparacion] Motor 3 has no AGEB backtest: its score is structural affinity, not an activity model

## 6. Procedencia

Registro `v3-claims-1` · contrato `6e586b3d9a0bd8b844d9cf5bc714f0e583fc2a6936f8cd5ca89de9cdb26afd93` · puntero `audits/09_final_claims_registry.json` · analysis_id del registro (hash de identidad de contenido): `acd556d569dfef3511f037479780acc3b54ed5f850aea2670b279bbf32b96422`.

| Cuaderno | Esquema | analysis_id |
|---|---|---|
| 01_ETL_Reproducible.ipynb | — | `sin analysis_id` |
| 02_EDA_y_Equidad.ipynb | v3-eda-2 | `d77ab12f05e9025f438921b34e8688e1a75628262964277af62250cdbafa8e67` |
| 03_Feature_Engineering.ipynb | v3-features-2 | `7565adda3bc4eb91514247d2a1213f0b1c934821e762ac9d3aae010031dbb05c` |
| 04_Diseno_Validacion_Backtest.ipynb | v3-backtest-4 | `ed7bf47bafab10ca2b3144e9d38f1d06e5024ee9ba6af353b37603324ad2ab32` |
| 05_Motor1_Baseline_Estadistico.ipynb | v3-motor1-3 | `879ae45d46b56fc5504c9817be154eee0e78ab0e13ddfdcef4435a58d32de29d` |
| 06_Motor2_Transformer_Honesto.ipynb | v3-motor2-3 | `4027385dff30ab79689dfdeb3c468260176080e83b1abd21f5c605efcce30a8f` |
| 07_Motor3_Grafo_y_Genetico.ipynb | v3-motor3-4 | `3aa53121add31f2ea2381259d0b44fa7f603fa758b4a731dcc08fd143b074b87` |
| 08_Evaluacion_Comparativa.ipynb | v3-comparacion-2 | `fdba0469ee4d2a42ce8bc5f75d53e95930207bfe3152faad5fbedd6ddc790f69` |


**Cierre.** Registro de afirmaciones sellado: toda cifra citable queda ligada a su publicación de origen.

<IPython.core.display.JSON object>